# Predictive Maintenance – Industrial Anomaly Detection
**Detecting failures in manufacturing equipment before they happen**

Relevant to BMW Group's production facilities (Werk München, Werk Dingolfing, etc.)  
Unplanned machine downtime costs the automotive industry billions annually. Early anomaly detection in sensor streams allows preventive maintenance — keeping production lines running.

**Goal:** Detect anomalous sensor readings in industrial machinery using unsupervised and supervised ML.

---

## 1. Setup & Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# ---------------------------------------------------------------
# Synthetic sensor stream: industrial robot arm / welding station
# Simulates 30 days of continuous sensor data at 10-minute intervals
# ---------------------------------------------------------------
N_NORMAL  = 4000
N_ANOMALY = 120   # ~3% anomaly rate

# Normal operating conditions
normal = pd.DataFrame({
    'vibration_mm_s':    np.random.normal(2.5, 0.4, N_NORMAL),
    'temperature_c':     np.random.normal(68.0, 3.0, N_NORMAL),
    'current_amp':       np.random.normal(12.0, 0.8, N_NORMAL),
    'pressure_bar':      np.random.normal(5.5, 0.3, N_NORMAL),
    'rotation_rpm':      np.random.normal(1450, 30, N_NORMAL),
    'label': 0  # 0 = normal
})

# Anomalous conditions (bearing wear, overheating, pressure drop, etc.)
anomalies = pd.DataFrame({
    'vibration_mm_s':    np.random.normal(8.0, 2.0, N_ANOMALY),   # high vibration
    'temperature_c':     np.random.normal(95.0, 8.0, N_ANOMALY),  # overheating
    'current_amp':       np.random.normal(18.5, 2.5, N_ANOMALY),  # overcurrent
    'pressure_bar':      np.random.normal(2.5, 0.8, N_ANOMALY),   # pressure drop
    'rotation_rpm':      np.random.normal(1200, 80, N_ANOMALY),   # speed deviation
    'label': 1  # 1 = anomaly
})

df = pd.concat([normal, anomalies], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Simulate timestamps
df['timestamp'] = pd.date_range(start='2024-01-01', periods=len(df), freq='10min')

print(f'Dataset: {len(df)} samples | Normal: {(df.label==0).sum()} | Anomalies: {(df.label==1).sum()}')
df.head()

## 2. Sensor Data Visualization

In [ ]:
sensors = ['vibration_mm_s', 'temperature_c', 'current_amp', 'pressure_bar']
fig, axes = plt.subplots(len(sensors), 1, figsize=(14, 10), sharex=True)

# Show first 500 samples for clarity
subset = df.iloc[:500].copy()

for ax, sensor in zip(axes, sensors):
    normal_mask  = subset['label'] == 0
    anomaly_mask = subset['label'] == 1
    ax.plot(subset.loc[normal_mask, 'timestamp'],  subset.loc[normal_mask, sensor],
            color='steelblue', linewidth=0.6, alpha=0.8, label='Normal')
    ax.scatter(subset.loc[anomaly_mask, 'timestamp'], subset.loc[anomaly_mask, sensor],
               color='red', s=30, zorder=5, label='Anomaly')
    ax.set_ylabel(sensor, fontsize=9)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
plt.xticks(rotation=30)
plt.suptitle('Industrial Sensor Stream — Anomalies Highlighted in Red', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('sensor_stream.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Anomaly Detection Models

In [ ]:
FEATURES = ['vibration_mm_s', 'temperature_c', 'current_amp', 'pressure_bar', 'rotation_rpm']
X = df[FEATURES].values
y_true = df['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Isolation Forest ---
iso = IsolationForest(contamination=0.03, n_estimators=200, random_state=42)
y_iso = iso.fit_predict(X_scaled)
y_iso = (y_iso == -1).astype(int)  # -1 = anomaly → 1

# --- Local Outlier Factor ---
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.03)
y_lof = lof.fit_predict(X_scaled)
y_lof = (y_lof == -1).astype(int)

print('=== Isolation Forest ===')
print(classification_report(y_true, y_iso, target_names=['Normal','Anomaly']))

print('=== Local Outlier Factor ===')
print(classification_report(y_true, y_lof, target_names=['Normal','Anomaly']))

## 4. Confusion Matrices & Score Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(axes,
    [y_iso, y_lof],
    ['Isolation Forest', 'Local Outlier Factor']):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Anomaly'],
                yticklabels=['Normal','Anomaly'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Anomaly Score Timeline (Isolation Forest)

In [ ]:
anomaly_scores = -iso.decision_function(X_scaled)  # higher = more anomalous
threshold = np.percentile(anomaly_scores, 97)

subset = df.iloc[:500].copy()
scores_subset = anomaly_scores[:500]

plt.figure(figsize=(14, 4))
plt.plot(subset['timestamp'], scores_subset, color='steelblue', linewidth=0.7, alpha=0.8)
plt.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, label=f'Alert threshold ({threshold:.3f})')
plt.fill_between(subset['timestamp'], scores_subset, threshold,
                 where=(scores_subset > threshold), alpha=0.4, color='red', label='Detected anomaly')
plt.xlabel('Time')
plt.ylabel('Anomaly Score')
plt.title('Real-time Anomaly Score — Production Sensor Stream', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('anomaly_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print('=== RESULTS SUMMARY ===')
for name, y_pred in [('Isolation Forest', y_iso), ('Local Outlier Factor', y_lof)]:
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    f = f1_score(y_true, y_pred)
    print(f'{name:25s} | Precision: {p:.3f} | Recall: {r:.3f} | F1: {f:.3f}')

print()
print('Key insights:')
print('  - Isolation Forest detects anomalies with high recall — critical for safety-first environments')
print('  - Multi-sensor fusion is more robust than single-sensor thresholds')
print('  - Real-time scoring enables proactive maintenance scheduling')
print()
print('BMW relevance:')
print('  - Applicable to welding robots, press lines, conveyor systems, CNC machines')
print('  - Reduces unplanned downtime and prevents costly production stops')
print('  - Foundation for BMW Plant Intelligence / Industry 4.0 initiatives')